# Notebook 2: GEDI UOI Signal Generation (GEE)

This notebook computes the raw Understory Openness Index (UOI) from GEDI L2B data.
It follows a strict modular structure:
1. Setup & Configuration
2. Methodological Logic (Functions)
3. Unit Tests
4. Execution (Asset Export)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee
import geemap

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")

# Output destination
ASSET_ROOT = 'users/JakeWilliams844/DefaunationFromSpace'

# Study Regions
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
STUDY_REGION = ee.FeatureCollection([
    ee.Feature(CONGO_BBOX, {'basin': 'Congo'}),
    ee.Feature(AMAZON_BBOX, {'basin': 'Amazon'})
])

# Scales (meters)
SCALES = list(range(5000, 105000, 5000))

# Datasets
GEDI_L2B = 'LARSE/GEDI/GEDI02_B_002_MONTHLY'
FOREST_MASK = 'projects/JRC/TMF/v1_2024/TransitionMap_MainClasses'
FOREST_CLASS = 10
FOREST_COVER_THRESHOLD = 0.95
SRTM = 'USGS/SRTMGL1_003'

# Date range
START_DATE = '2020-01-01'
END_DATE = '2023-12-31'

print("✓ Configuration loaded.")

In [ ]:
# =============================================================================
# BLOCK 2: METHODOLOGICAL LOGIC (FUNCTIONS)
# =============================================================================

def load_base_masks():
    """Loads JRC TMF intact forest mask and SRTM topographic quality masks."""
    # Intact Forest
    tmf = ee.Image(FOREST_MASK)
    forest_mask = tmf.eq(FOREST_CLASS)
    
    # Topography (Elevation < 1000m, Slope < 10 deg)
    srtm = ee.Image(SRTM)
    elev = srtm.select('elevation')
    slope = ee.Terrain.slope(srtm)
    
    topo_mask = elev.lt(1000).And(slope.lt(10))
    
    # Combined native mask
    combined_mask = forest_mask.updateMask(topo_mask)
    return combined_mask, forest_mask

def compute_native_uoi(combined_mask):
    """Calculates native resolution UOI and observation counts from GEDI."""
    # Load GEDI and filter by date
    gedi = ee.ImageCollection(GEDI_L2B).filterDate(START_DATE, END_DATE)
    
    def calc_uoi(img):
        pai = img.select('pai')
        pavd_z0 = img.select('pavd_z0')
        
        # UOI = 1 - (pavd_z0 / pai)
        uoi = ee.Image(1).subtract(pavd_z0.divide(pai))
        # Clip to [0, 1] range to handle anomalies
        uoi = uoi.clamp(0, 1).rename('UOI')
        return uoi.updateMask(combined_mask)
    
    # Map UOI calculation over collection
    uoi_col = gedi.map(calc_uoi)
    
    # Calculate mean UOI and count N (number of valid footprint observations)
    mean_uoi = uoi_col.mean().rename('UOI_mean')
    count_n = uoi_col.count().rename('N')
    
    # Stack the bands
    native_stack = ee.Image.cat([mean_uoi, count_n])
    native_proj = gedi.first().projection()
    
    return native_stack, native_proj

def build_gedi_asset(scale):
    """Aggregates UOI to the target scale using reduceResolution."""
    combined_mask, forest_mask = load_base_masks()
    native_stack, native_proj = compute_native_uoi(combined_mask)
    
    # Aggregate forest mask to enforce 95% intactness at scale
    forest_fraction = forest_mask.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).reproject(
        crs=native_proj,
        scale=scale
    )
    scale_mask = forest_fraction.gte(FOREST_COVER_THRESHOLD)
    
    # Aggregate UOI (mean) and N (sum)
    # We need to use separate reducers
    agg_uoi = native_stack.select('UOI_mean').reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=65535
    ).reproject(
        crs=native_proj,
        scale=scale
    )
    
    agg_n = native_stack.select('N').reduceResolution(
        reducer=ee.Reducer.sum(),
        maxPixels=65535
    ).reproject(
        crs=native_proj,
        scale=scale
    )
    
    # Stack and mask
    final_asset = ee.Image.cat([agg_uoi, agg_n]).updateMask(scale_mask)
    
    return final_asset, native_proj

print("✓ Methodological functions loaded.")

In [ ]:
# =============================================================================
# BLOCK 3: UNIT TESTS
# =============================================================================

def run_unit_tests():
    print("Running Unit Tests for GEDI UOI generation...")
    test_scale = 50000
    try:
        gedi_asset, proj = build_gedi_asset(test_scale)
        
        # Test 1: Image type
        assert isinstance(gedi_asset, ee.Image), "Output is not an ee.Image"
        
        # Test 2: Band names
        bands = gedi_asset.bandNames().getInfo()
        assert len(bands) == 2, f"Expected 2 bands, got {len(bands)}"
        assert 'UOI_mean' in bands, "Missing UOI_mean band"
        assert 'N' in bands, "Missing N band"
        
        print("✓ All unit tests passed!")
    except AssertionError as e:
        print(f"✗ Unit Test Failed: {e}")
    except Exception as e:
        print(f"✗ Unexpected Error during tests: {e}")

# Execute tests
run_unit_tests()

In [ ]:
# =============================================================================
# BLOCK 4: EXECUTION (ASSET EXPORT)
# =============================================================================

def export_all_scales(dry_run=True):
    tasks = []
    print(f"Configuring export tasks for {len(SCALES)} scales...")
    
    for scale in SCALES:
        gedi_asset, proj = build_gedi_asset(scale)
        
        task = ee.batch.Export.image.toAsset(
            image=gedi_asset,
            description=f'GEDI_{scale}_Export',
            assetId=f'{ASSET_ROOT}/GEDI_{scale}',
            region=STUDY_REGION.geometry(),
            scale=scale,
            crs=proj,
            maxPixels=1e13
        )
        tasks.append(task)
        
        if not dry_run:
            task.start()
            
    print(f"✓ Configured {len(tasks)} export tasks.")
    if dry_run:
        print("DRY RUN: Tasks created but not started. Call export_all_scales(dry_run=False) to begin processing on GEE servers.")
    else:
        print("Tasks started! Monitor progress in the GEE Code Editor or via ee.batch.Task.list()")

# To execute the exports, set dry_run=False
export_all_scales(dry_run=True)